# 04 — Held-out trust check and v6 stress test

This notebook computes causal truth for the already-frozen answer-type-matched controls, then performs the model-free joins and grouped inference. It evaluates old and hard-control binary AUC, the decisive engine-only graded correlation, raw-score distributions, and the final `ARTIFACT (partial)` decision. It imports no cheap estimator code.

**Outputs:** `artifacts/final/04_hard_causal.json` and `04_evaluation.json`.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / 'artifacts' / 'final'

from src.causal_read import (
    compute_hard_dashboard_causal_rows,
    summarize_hard_dashboard_causal_sanity,
)
from src.evaluation import assemble_final_metrics
from src.jlens_interface import load_model, release_model

def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

dataset = load_json(ARTIFACTS / '01_dataset.json')
base_causal_path = ARTIFACTS / '02_causal.json'
base_causal = load_json(base_causal_path)
base_cheap = load_json(ARTIFACTS / '03_cheap.json')
hard_manifest_path = ARTIFACTS / '03_hard_manifest.json'
hard_manifest = load_json(hard_manifest_path)
hard_cheap = load_json(ARTIFACTS / '03_hard_cheap.json')
verified_hard = [row for row in hard_manifest['rows'] if row['verification_status'] == 'VERIFIED_HARD']

bundle = load_model()
try:
    hard_rows = compute_hard_dashboard_causal_rows(
        bundle,
        verified_hard,
        base_causal['rows'],
        layer=int(hard_manifest['selection']['layer']),
    )
finally:
    del bundle
    release_model()

hard_sanity = summarize_hard_dashboard_causal_sanity(hard_rows)
hard_causal = {
    'schema_version': 'read-stress-v6-hard-dashboard-causal-v1',
    'model': hard_manifest['model'],
    'selected_layer': hard_manifest['selection']['layer'],
    'position_rule': hard_manifest['selection']['position_rule'],
    'source_hard_manifest': {
        'path': str(hard_manifest_path.relative_to(ROOT)),
        'sha256': sha256(hard_manifest_path),
    },
    'frozen_causal_truth': {
        'path': str(base_causal_path.relative_to(ROOT)),
        'sha256': sha256(base_causal_path),
        'engine_recomputed': False,
    },
    'causal_sanity': hard_sanity,
    'rows': hard_rows,
}
hard_causal_path = ARTIFACTS / '04_hard_causal.json'
write_json(hard_causal_path, hard_causal)

final_metrics = assemble_final_metrics(
    dataset,
    base_causal,
    base_cheap,
    hard_manifest,
    hard_cheap,
    hard_causal,
    n_bootstrap=10_000,
    seed=1729,
)
evaluation_path = ARTIFACTS / '04_evaluation.json'
write_json(evaluation_path, final_metrics)
print(json.dumps({
    'hard_causal_sanity': hard_sanity,
    'decision': final_metrics['decision'],
    'evaluation_sha256': sha256(evaluation_path),
}, indent=2))


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{
  "hard_causal_sanity": {
    "n_pairs": 77,
    "engine_C_median": 0.9127144298688193,
    "engine_abs_C_median": 0.9127144298688193,
    "hard_dashboard_C_median": 0.0012345679012345677,
    "hard_dashboard_abs_C_median": 0.00646551724137931,
    "hard_dashboard_C_min": -0.03321256038647343,
    "hard_dashboard_C_max": 0.017278617710583154,
    "hard_dashboard_sharp_directional_disagreements": 0,
    "engine_large_gate": true,
    "hard_dashboard_near_zero_gate": true,
    "status": "PASS"
  },
  "decision": {
    "decision": "ARTIFACT (partial)",
    "decision_one_line": "ARTIFACT (partial): READ_IG survives the answer-type-matched control, but has no positive graded association within engines (rho=-0.179, 95% CI [-0.431, 0.126]); the perfect binary separation is not evidence of a graded causal-use meter.",
    "binary_detector": "SUPPORTED",
    "graded_meter": "NOT_SUPPORTED",
    "stress_test_label": "ARTIFACT (partial)",
    "decision_rule": {
      "confirmed_requires_both": 